Eventually, you'll want to take advantage of the automatization that SCOPE enables. In this tutorial, we will prepare a more advanced workflow.  
Make sure you read and understood Tutorial 4 before starting with this one

In [ ]:
import os
import scope

In [ ]:
## Path of the data folder. It should be "os.path.abspath('.')+'/Data/Tutorial_4/"
tutorial_folder = os.path.abspath('.')+'/Data/Tutorial_5/'

# Example:

## 1.1) Create Systems: Water, Benzene, Formaldehyde

In [ ]:
## In this example, we create three systems, each with 4 .xyz sources.
from scope.classes_system import System
water_sys   = System("water")
benz_sys    = System("benzene")
form_sys    = System("formaldehyde")

## We set the paths. See Tutorial_4 for more options and information
list_of_systems = []
for tsys in [water_sys, benz_sys, form_sys]:
    tsys.sources_path = f"{tutorial_folder}Sources/{tsys.name}/"
    tsys.calcs_path   = f"{tutorial_folder}Calcs/{tsys.name}/"
    tsys.sys_path     = f"{tutorial_folder}Systems/{tsys.name}/"
    tsys.sys_file     = f"{tutorial_folder}Systems/{tsys.name}/{tsys.name}.npy"
    list_of_systems.append(tsys)

In [ ]:
## We stored 3 systems in a list, with very little information, only the paths. For instance:
print(list_of_systems[0])

## 1.2) Import Molecules in Source Folder 

In [ ]:
# Currently, SCOPE is designed so each system has its own folder with sources. In this example, the sources are .xyz files
# Unfortunately, there is no universal way of creating systems. Each problem in computational chemistry faces its own system structure. 
# In the SCO add-on, the original data is in a cell2mol CELL format, so a new function had to be created to import molecules from there. 
# New add-ons are being prepared, each one with its own form of importing systems, and arranging molecules within those systems.

# For the moment, the simplest way to import sources is when those are in an XYZ format.
for tsys in list_of_systems: 
    tsys.load_multiple_xyz(tsys.sources_path)
    tsys.save()

## If you try to re-run this cell, you'll receive warnings that you're trying to add new sources with existing names. 
## This would be very dangerous. 

In [ ]:
## We now added the molecules as sources. For instance:
print(list_of_systems[0])

In [ ]:
for sou in tsys.sources:
    #print(sou.name, sou.sys_name)
    print(sou)

## 2) Create a Job File - Single point with PBE

In [ ]:
from scope.classes_input import *

In [ ]:
## Job Specs can be created from a file, or from a string. 
## This is an example input read as a string. We will soon go through the different parts
## First, notice that there are four sections, initiated with '&', and terminated with '/'.
## Sections can be empty as in the example below (see &options) or even absent. In those cases, default values will be added
## Also, notice that the values are either introduced as strings 'X', lists [], booleans, or integers/floats...
## Please respect the notation used for each type of variable

test_input = """
&environment
   max_jobs         = 12
   max_procs        = 20
   requested_procs  = 1
/

&options
/

&job_data
   branch       = 'Isolated'
   workflow     = 'all'
   job_setup    = 'regular'
   suffix       = 'scf'
   keyword      = 'pbe scf'

   istate       = 'initial'
   fstate       = 'initial'

   hierarchy    = 1
   requisites   = []
   constrains   = ['self']
   must_be_good = False
/

&qc_data
   software     = 'g16'
   jobtype      = 'scf'
   functional   = 'pbe'
   basis        = 'sto-3g'
   is_grimme    = True
/"""

In [ ]:
# IMPORTANT. Notice that, when running the 'scope_run_job' script. The job file (-j arg) must be an actual file, not a string as in the cell above
# Thus, we save this text to an actual file, so we can call it later
from scope.read_write import save_text
file_name = ''.join([tutorial_folder,"test_job.job"])
save_text(test_input, file_name)

## 3) Run

Once the systems and job_file are prepared, we can submit the job in the appropriate computation cluster.

This computation cluster should have **Gaussian 16** installed, and a **configured environment**. 
You can give it any name, but for the sake of simplicity, here we will call it **"scope_env_cluster.npy"**  

Then, we can use the 'scope_run_job' script. If you didn't change the SYSTEM and JOB_FILE paths used in this tutorial, it should be like that:

1) Go to the Tutorial_5 folder inside a computation cluster, and:
2) ```bash
   scope_run_job -n scope_env_cluster.npy -s Systems/water/water.npy -j test_job.job
   scope_run_job -n scope_env_cluster.npy -s Systems/benzene/benzene.npy -j test_job.job
   scope_run_job -n scope_env_cluster.npy -s Systems/formaldehyde/formaldehyde.npy -j test_job.job
   ```

In [ ]:
## The environment of the computation cluster (scope_env_cluster.npy) will automatically update the paths stored in the system.
## It will use the function:
help(tsys.set_paths_from_environment)